In [1]:
import pandas as pd

1. Forecasted Demand Distribution
    - Using clustering output:Hub, Shift, Employees
2. Vehicle Capacity Analysis
    - For each group: Example: Saravanampatti 04:30, 177 employees    
    - How many: SEDAN, SUV, VAN are needed?
3. Utilization Analysis
    - Example:177 employees, 22 Vans = 176 seats, Utilization = 100.6%
    - 23 Vans = 184 seats, Utilization = 96.2%
4. Cost Analysis
    - Using pricing_engine.    
5. Escort Analysis
    - We discovered: 368 escort-required employees, That's huge. This may affect: Vehicle type, Vendor selection, Cost    

Optimize: Cost + Utilization + Safety + Vendor Quality

EDA ---> prototype ---> allocation strategy decision

# Business Objective
Goal:

Given forecasted employee demand,
determine:

1. Vehicle count
2. Vehicle type mix
3. Capacity utilization
4. Estimated transport cost
5. Escort requirement
6. Vendor requirement

while balancing:

- Cost
- Utilization
- Safety
- SLA

In [2]:
# load fleet registry
fleet_df = pd.read_csv(
    r"D:\mlprojects\trns_proj\data\fleet_registry.csv"
)

fleet_df.head()

,vehicle_id,vendor_name,vehicle_type,seating_capacity,fuel_type,gps_enabled,panic_button_enabled,safety_rating,active
0,FT-SUV,FastTrack Mobility,SUV,6,CNG,True,True,8.8,True
1,FT-TT,FastTrack Mobility,TEMPO_TRAVELLER,14,DIESEL,True,True,8.8,True
2,FT-MAZ,FastTrack Mobility,MAZDA,20,DIESEL,True,True,8.8,True
3,SR-SUV,SecureRide Logistics,SUV,6,CNG,True,True,9.5,True
4,SR-TT,SecureRide Logistics,TEMPO_TRAVELLER,14,DIESEL,True,True,9.5,True


In [3]:
fleet_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   vehicle_id            9 non-null      str    
 1   vendor_name           9 non-null      str    
 2   vehicle_type          9 non-null      str    
 3   seating_capacity      9 non-null      int64  
 4   fuel_type             9 non-null      str    
 5   gps_enabled           9 non-null      bool   
 6   panic_button_enabled  9 non-null      bool   
 7   safety_rating         9 non-null      float64
 8   active                9 non-null      bool   
dtypes: bool(3), float64(1), int64(1), str(4)
memory usage: 930.0 bytes


In [4]:
fleet_df.groupby(
    ["vendor_name", "vehicle_type"]
).size()

vendor_name           vehicle_type   
CityCommute Services  MAZDA              1
                      SUV                1
                      TEMPO_TRAVELLER    1
FastTrack Mobility    MAZDA              1
                      SUV                1
                      TEMPO_TRAVELLER    1
SecureRide Logistics  MAZDA              1
                      SUV                1
                      TEMPO_TRAVELLER    1
dtype: int64

In [5]:
fleet_df.describe(
    include="all"
)

,vehicle_id,vendor_name,vehicle_type,seating_capacity,fuel_type,gps_enabled,panic_button_enabled,safety_rating,active
count,9,9,9,9.000000,9,9,9,9.000000,9
unique,9,3,3,NaN,2,1,2,NaN,1
top,FT-SUV,FastTrack Mobility,SUV,NaN,DIESEL,True,True,NaN,True
freq,1,3,3,NaN,6,9,6,NaN,9
mean,NaN,NaN,NaN,13.333333,NaN,NaN,NaN,8.800000,NaN
std,NaN,NaN,NaN,6.082763,NaN,NaN,NaN,0.606218,NaN
min,NaN,NaN,NaN,6.000000,NaN,NaN,NaN,8.100000,NaN
25%,NaN,NaN,NaN,6.000000,NaN,NaN,NaN,8.100000,NaN
50%,NaN,NaN,NaN,14.000000,NaN,NaN,NaN,8.800000,NaN
75%,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,9.500000,NaN


## Fleet Inventory EDA
How many vehicles?

How many sedans?

How many SUVs?

How many vans?

Available capacity?

In [6]:
fleet_df[
    "vehicle_type"
].value_counts()

vehicle_type
SUV                3
TEMPO_TRAVELLER    3
MAZDA              3
Name: count, dtype: int64

In [7]:
fleet_df.groupby(
    "vehicle_type"
)[
    "seating_capacity"
].agg(
    [
        "count",
        "mean",
        "sum",
    ]
)

,count,mean,sum
vehicle_type,,,
MAZDA,3,20.0,60
SUV,3,6.0,18
TEMPO_TRAVELLER,3,14.0,42


In [8]:
fleet_df[
    "vendor_name"
].value_counts()

vendor_name
FastTrack Mobility      3
SecureRide Logistics    3
CityCommute Services    3
Name: count, dtype: int64

In [9]:
pd.crosstab(
    fleet_df["vendor_name"],
    fleet_df["vehicle_type"]
)

vehicle_type,MAZDA,SUV,TEMPO_TRAVELLER
vendor_name,,,
CityCommute Services,1,1,1
FastTrack Mobility,1,1,1
SecureRide Logistics,1,1,1


In [10]:
demand_df = pd.DataFrame(
    {
        "hub": [
            "Ganapathy",
            "Ganapathy",
            "Hopes",
            "Hopes",
            "Saravanampatti",
            "Saravanampatti",
            "Singanallur",
            "Singanallur",
            "Thudiyalur",
            "Thudiyalur",
        ],

        "transport_shift": [
            "03:30",
            "04:30",
            "03:30",
            "04:30",
            "03:30",
            "04:30",
            "03:30",
            "04:30",
            "03:30",
            "04:30",
        ],

        "employee_count": [
            33,
            55,
            60,
            79,
            126,
            177,
            73,
            141,
            77,
            75,
        ]
    }
)

In [11]:
# if use mazda only Capacity = 20
import math

demand_df["van_count"] = (
    demand_df["employee_count"]
    .apply(
        lambda x:
        math.ceil(x / 20)
    )
)

demand_df["available_seats"] = (
    demand_df["van_count"] * 20
)

demand_df["utilization_pct"] = (
    demand_df["employee_count"]
    /
    demand_df["available_seats"]
) * 100

demand_df

,hub,transport_shift,employee_count,van_count,available_seats,utilization_pct
0,Ganapathy,03:30,33,2,40,82.500000
1,Ganapathy,04:30,55,3,60,91.666667
2,Hopes,03:30,60,3,60,100.000000
3,Hopes,04:30,79,4,80,98.750000
4,Saravanampatti,03:30,126,7,140,90.000000
5,Saravanampatti,04:30,177,9,180,98.333333
6,Singanallur,03:30,73,4,80,91.250000
7,Singanallur,04:30,141,8,160,88.125000
8,Thudiyalur,03:30,77,4,80,96.250000
9,Thudiyalur,04:30,75,4,80,93.750000


- for mazda only strategy average utilization is approximately: 93% which is outstanding.

In [12]:
'''
# Allocation Strategy version 1
if employee_count >= 40:
    vehicle_type = "MAZDA"

elif employee_count >= 15:
    vehicle_type = "TEMPO_TRAVELLER"

else:
    vehicle_type = "SUV"
'''    



'\n# Allocation Strategy version 1\nif employee_count >= 40:\n    vehicle_type = "MAZDA"\n\nelif employee_count >= 15:\n    vehicle_type = "TEMPO_TRAVELLER"\n\nelse:\n    vehicle_type = "SUV"\n'

# Mixed fleet strategy

In [13]:
# vehicle master
VEHICLES = {
    "SUV": 6,
    "TEMPO_TRAVELLER": 14,
    "MAZDA": 20,
}

In [14]:
# Prototype Optimizer for now don't use OR-Tools. Use brute force.
def best_vehicle_mix(
    demand: int,
):

    best_solution = None

    best_unused = float("inf")

    for mazda in range(0, 20):

        for tt in range(0, 20):

            for suv in range(0, 20):

                seats = (
                    mazda * 20
                    +
                    tt * 14
                    +
                    suv * 6
                )

                if seats < demand:
                    continue

                unused = (
                    seats - demand
                )

                if unused < best_unused:

                    best_unused = unused

                    best_solution = {

                        "demand": demand,

                        "mazda": mazda,

                        "tempo_traveller": tt,

                        "suv": suv,

                        "total_seats": seats,

                        "unused_seats": unused,
                    }

    return best_solution

In [15]:
best_vehicle_mix(
    177
)

{'demand': 177,
 'mazda': 0,
 'tempo_traveller': 5,
 'suv': 18,
 'total_seats': 178,
 'unused_seats': 1}

In [16]:
results = []

for demand in [
    33,
    55,
    60,
    73,
    75,
    77,
    79,
    126,
    141,
    177,
]:

    results.append(
        best_vehicle_mix(
            demand
        )
    )

allocation_df = pd.DataFrame(
    results
)

allocation_df

,demand,mazda,tempo_traveller,suv,total_seats,unused_seats
0,33,0,2,1,34,1
1,55,0,1,7,56,1
2,60,0,0,10,60,0
3,73,0,1,10,74,1
4,75,0,2,8,76,1
5,77,0,0,13,78,1
6,79,0,1,11,80,1
7,126,0,3,14,126,0
8,141,0,2,19,142,1
9,177,0,5,18,178,1


- we calculated based on employee only not included security
- Current optimization minimizes unused_seats only.
- Real enterprise objective is: cost + unused_seats + safety + escort compliance
- For mazda only: 177 employees we need 9 Mazda, 180 seats, 98.3% utilization
- For mixed fleet 177 employees found- 5 TT + 18 SUV, 178 seats, 99.4% utilization


In [19]:
employees_df = pd.read_csv(
    r'D:\mlprojects\trns_proj\data\employees_after_churn.csv'
)
employees_df.head(5)

,employee_id,gender,hub,pickup_hub,uses_company_transport,transport_shift,login_shift,transport_eligibility,extension_category,shift_logout,predicted_extension_minutes,safety_priority_score,requires_security_escort,home_distance_km,home_lat,home_lon
0,1,Female,Hopes,Hopes,False,NON_TRANSPORT,NON_TRANSPORT,FULL_HOME_DROP,NO_EXTENSION,2025-01-01 18:30:00,0,4.0,False,7.38,11.007393,77.022740
1,3,Female,Ganapathy,Ganapathy,False,NON_TRANSPORT,NON_TRANSPORT,FULL_HOME_DROP,NO_EXTENSION,2025-01-01 21:30:00,0,4.0,False,3.87,11.036700,76.984871
2,4,Female,Saravanampatti,Saravanampatti,True,04:30,19:30,FULL_HOME_DROP,NO_EXTENSION,2025-01-01 04:30:00,0,7.0,True,14.11,11.132525,77.008819
3,5,Male,Hopes,Hopes,False,NON_TRANSPORT,NON_TRANSPORT,FULL_HOME_DROP,NO_EXTENSION,2025-01-01 18:30:00,0,1.0,False,10.44,11.031401,77.050282
4,6,Male,Thudiyalur,Thudiyalur,False,NON_TRANSPORT,NON_TRANSPORT,FULL_HOME_DROP,NO_EXTENSION,2025-01-01 21:30:00,0,1.0,False,11.74,11.122190,76.962961


In [21]:
escort_summary = (
    employees_df
    .groupby(
        [
            "hub",
            "transport_shift"
        ]
    )[
        "requires_security_escort"
    ]
    .sum()
    .reset_index()
)
escort_summary

,hub,transport_shift,requires_security_escort
0,Ganapathy,03:30,10
1,Ganapathy,04:30,25
2,Ganapathy,NON_TRANSPORT,13
3,Hopes,03:30,24
4,Hopes,04:30,21
5,Hopes,NON_TRANSPORT,17
6,Saravanampatti,03:30,59
7,Saravanampatti,04:30,73
8,Saravanampatti,NON_TRANSPORT,36
9,Singanallur,03:30,33


In [22]:
escort_df = (
    employees_df
    .groupby(
        [
            "hub",
            "transport_shift"
        ]
    )
    .agg(
        total_employees=(
            "employee_id",
            "count"
        ),
        escort_required=(
            "requires_security_escort",
            "sum"
        )
    )
    .reset_index()
)

escort_df

,hub,transport_shift,total_employees,escort_required
0,Ganapathy,03:30,33,10
1,Ganapathy,04:30,55,25
2,Ganapathy,NON_TRANSPORT,79,13
3,Hopes,03:30,60,24
4,Hopes,04:30,79,21
5,Hopes,NON_TRANSPORT,149,17
6,Saravanampatti,03:30,126,59
7,Saravanampatti,04:30,177,73
8,Saravanampatti,NON_TRANSPORT,289,36
9,Singanallur,03:30,73,33
